### Engenharia de Features

##### Nessa etapa, transformamos o que encontramos na última EDA em colunas numéricas que o modelo consiga utilizar.

In [1]:
import pandas as pd

Carrega e converte a data

In [3]:
df_final = pd.read_csv("../data/processed/base_unificada.csv")

df_final["order_purchase_timestamp"] = pd.to_datetime(df_final["order_purchase_timestamp"])

Aqui, vamos construir a feature que a última EDA revelou dos pedidos por vendedor, a taxa histórica de atraso de cada um. Algumas exceções precisaram ser tratadas pra evitar vazamento.


In [4]:
pedidos_vendedor = df_final[["order_id","seller_id","order_purchase_timestamp","dias_atrasados"]].drop_duplicates("order_id")
pedidos_vendedor = pedidos_vendedor.sort_values("order_purchase_timestamp").reset_index(drop=True)

pedidos_vendedor["atrasado_flag"] = (pedidos_vendedor["dias_atrasados"] > 0).astype(int)

pedidos_vendedor["pedidos_anteriores"] = pedidos_vendedor.groupby("seller_id").cumcount()
pedidos_vendedor["atrasos_anteriores"] = pedidos_vendedor.groupby("seller_id")["atrasado_flag"].transform(lambda x: x.shift().cumsum())

pedidos_vendedor["taxa_atraso_vendedor"] = pedidos_vendedor["atrasos_anteriores"] / pedidos_vendedor["pedidos_anteriores"]

Só junta de volta a taxa calculada na base principal, cruzando por order_id 

In [5]:
df_final = df_final.merge(pedidos_vendedor[["order_id","taxa_atraso_vendedor"]], on="order_id", how="left")

Confirmamos o NaN para validar que a feature não vazou, esses 3499 são os primeiros pedidos de cada vendedor que ainda não possuem histórico anterior.

In [6]:
df_final["taxa_atraso_vendedor"].isna().sum()

np.int64(3499)

Salvando o resultado com essa nova feature, já pronto pra entrar na modelagem

In [7]:
df_final.to_csv("../data/processed/df_features.csv", index=False)

-----------------------------------------------------------------------------